# D1.1 · From alert queue to loop operator

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *AI for Security*

Builds on **[D1.0 · Start here — what AI for security operations means](https://spbreed.github.io/cyber-commons/lessons/D1.0.html)**.

| | |
|---|---|
| Tools used | Wazuh, OpenSearch, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Run a triage loop over Wazuh alerts and supervise by exception.

**Why a security engineer needs it.** Supervising by re-reading everything the loop did. The control it builds is: know what the loop must escalate and sample the rest.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The queue does not go away; it changes shape. Instead of triaging alerts you are supervising something that triages alerts, which is a different skill with a different quality bar and a much worse failure mode: confident, fast, and wrong at volume.

> **At CyberTravels.** The analyst on CyberTravels' alerts stops triaging and starts supervising something that triages — which is a different skill, with a worse failure mode: confident, fast, and wrong at volume.

## 2 · The framework

```
   before                          after
   +------------------+            +---------------------------+
   | alert -> analyst |            | alert -> loop -> analyst  |
   |          decides |            |          proposes  reviews|
   +------------------+            +---------------------------+
      100 alerts/day                  1000 alerts/day, 40 reviewed

   new failure mode: confident, fast, and wrong at volume
```

The classic SOC job is a queue: alerts arrive, an analyst reads each one,
decides, and moves on. The constraint is human attention, and it does not scale
— which is why tier-1 burnout and alert fatigue are structural rather than
cultural problems.

The agentic version replaces "read every alert" with "operate a loop that reads
every alert". The analyst's job becomes:

- deciding **what the loop is allowed to conclude** (the verifier, B2.0),
- deciding **what it may do about it** (the tool policy, A3.5),
- and handling the cases it escalates.

The skill that transfers is not triage speed. It is knowing which signals the
loop may believe — because a triage loop with a weak verifier closes true
positives at machine speed, and closing a true positive is silent.

## 3 · Where it breaks — closing a true positive is silent

Every triage decision has two error directions and they are not symmetric. Escalating a false positive costs an analyst ten minutes. **Closing a true positive costs you the incident**, and nothing tells you it happened.

## 4 · The control — the loop may close, but not silently

Three rules make an agentic triage loop safe to run, and none of them is about model quality.

## 5 · The procedure, as a skill

The skill scores a triage loop against ground truth, sweeps the confidence bar so the trade between analyst minutes and missed incidents is made explicitly, and adds the severity floor that no automatic closure may cross whatever its confidence.

### The skill — [`skills/detection/triage-loop-with-floor/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/triage-loop-with-floor/SKILL.md)

```yaml
name: triage-loop-with-floor
description: >-
  Run an alert triage loop against ground truth, measure the confusion matrix,
  and bound it with a confidence bar and a severity floor no automatic closure
  may cross. Use when automating triage, or when deciding what an agent may
  close on its own.
allowed-tools: Read, Grep, Glob
```

# The floor is the part that makes the loop deployable

A triage loop that matches ground truth on a sample is a promising loop. What
makes it something you can run is the pair of bounds around it: a confidence bar
that decides when it defers, and a severity floor it may never close through,
whatever its confidence.

## When to use this

Before an agent closes anything, and when tuning how much of a queue is
automated.

## Procedure

**1 — Score against ground truth, as a confusion matrix.** Escalated and closed,
against true and false. Accuracy alone hides the direction of the errors, and
only one direction matters here.

**2 — Sweep the confidence bar.** For each setting, record analyst minutes saved
and incidents missed. This is the trade being made, and it should be made
explicitly by whoever owns the queue rather than implicitly by a default.

**3 — Set the severity floor separately.** Any alert above it is escalated
regardless of confidence. A confident wrong closure on a critical alert is the
failure this exists to prevent, and no confidence threshold protects against it.

**4 — Check what the floor costs.** How many alerts it forces to a human per
day. If that number is above the reading budget, the floor is theatre and the
queue needs a different cut.

**5 — Record every automatic closure with its reason and confidence.** The loop
will be wrong sometimes; the question at review is whether you can find out how.

## Example

**Input** — the fixture committed at the top of [`scripts/triage_loop_with_floor.py`](scripts/triage_loop_with_floor.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
queue: 8 alerts, 4 true positives
   A-01 medium   impossible travel        dana@corp
   A-02 critical metadata service access  patch-agent
   A-03 medium   failed logins x40        svc-etl
   A-04 high     secret path read         patch-agent
   A-05 high     new admin group member   sam@corp
   A-06 low      port scan detected       scanner-01
   A-07 high     egress to unlisted host  triage-agent
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "confusion": {"tp": 0, "fp": 0, "tn": 0, "fn": 0},
  "confidence_sweep": [{"bar": 0.0, "analyst_minutes_saved": 0, "incidents_missed": 0}],
  "floor": {"severity": "str", "escalated_regardless": 0, "per_day": 0},
  "audit": {"closures_logged": true, "fields": ["reason", "confidence", "rule"]}
}
```

## Failure modes

- **Reporting accuracy.** The direction of the error is the whole question.
- **A floor set above the reading budget.** It routes to a person who will not
  read it.
- **Unlogged closures.** You cannot review what the loop decided.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/triage-loop-with-floor/scripts/triage_loop_with_floor.py
SCRIPT = "skills/detection/triage-loop-with-floor/scripts/triage_loop_with_floor.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The triage loop escalates 4 alerts and closes 4, matching ground truth on all 8. Lowering the confidence bar trades analyst minutes against missed incidents. The severity floor converts any high or critical closure into an escalation, and the closure sampling routes a fraction of routine closures to a human for quality measurement.

## Your turn

Ask your SOC one question: when an incident is confirmed, does anyone check whether an earlier alert about it was closed? If nobody does, you have no measurement of your false-negative rate — with or without an agent.

---

**Next → [D1.2 · Context that makes triage work](https://spbreed.github.io/cyber-commons/lessons/D1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*